In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask.array as da
import dask
import zarr
import xarray as xr
import cftime
import os

In [6]:
# ============== LOAD LLC ==============
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
    #     'name': 'strides=1,ckpt_4',
    #     'key': 'emulator_1',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=1,ckpt_4/predictions_4d.zarr',
    #     'desc': 'strides=1'
    # },
    # {
    #     'name': 'strides=1,ckpt_8',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=1,ckpt_8/predictions_4d.zarr',
    #     'desc': 'strides=1'
    # },
        # {
        'name': 'data,temporal,1,3,ckpt_6',
        'key': 'emulator_3',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-19-eval:Samudra_LLC:strides=[data,temporal,1,3],ckpt_6/predictions_4d.zarr',
        'desc': 'strides=(data,temporal,1,3)'
    },
    {
        'name': 'strides=3,ckpt_12',
        'key': 'emulator_4',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=3,ckpt_12/predictions_4d.zarr',
        'desc': 'strides=3'
    },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    for gv in grid_vars:
        patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded data,temporal,1,3,ckpt_6: strides=(data,temporal,1,3)
Loaded strides=3,ckpt_12: strides=3
LLC subset to 16 common times

=== Setup complete: LLC + 2 emulators ===
  data,temporal,1,3,ckpt_6 (emulator_3)
  strides=3,ckpt_12 (emulator_4)


In [7]:
selected_time_range = [0, 16]   # inclusive indices
stepping = 1                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:16
Stepping = 1
Final synchronized length = 16
LLC now has 16 times
data,temporal,1,3,ckpt_6 (emulator_3) now has 16 times
strides=3,ckpt_12 (emulator_4) now has 16 times


In [ ]:
vars = ['Theta', 'Salt', 'U', 'V']

out_dir = 'figs/prognostic_var_comparison'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'Prediction_Replication_Erors.png')

n_times = llc_patch.sizes['time']
time_indices = np.arange(1, n_times)

fig, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(10, 10),
    dpi=200,
    sharex=True
)

def as_dask(x):
    return x.data if isinstance(x.data, da.Array) else da.from_array(x.data)

def safe_norm(num, den):
    return da.where((den == 0) | da.isnan(den), np.nan, num / den)

for row, var in enumerate(vars):
    print(f"Generating plots for {var}...")

    # ==================================================
    # Surface arrays: expected shape = time, y, x
    # ==================================================
    llc_surf = as_dask(llc_patch[var].isel(k=0))

    llc_t1_surf = llc_surf[:-1, :, :]
    llc_t2_surf = llc_surf[1:, :, :]

    baseline_surf = da.nanmean(
        da.abs(llc_t2_surf - llc_t1_surf),
        axis=(-2, -1)
    )

    ax = axes[row, 0]

    for emu_name, emu_key in emulator_info:
        emu_surf = as_dask(emulator_patches[emu_key][var].isel(k=0))
        emu_t2_surf = emu_surf[1:, :, :]

        Pe_surf = da.nanmean(
            da.abs(llc_t2_surf - emu_t2_surf),
            axis=(-2, -1)
        )

        Re_surf = da.nanmean(
            da.abs(llc_t1_surf - emu_t2_surf),
            axis=(-2, -1)
        )

        Pe_norm, Re_norm = dask.compute(
            safe_norm(Pe_surf, baseline_surf),
            safe_norm(Re_surf, baseline_surf)
        )

        ax.plot(time_indices, Pe_norm, marker='o', linewidth=1.5,
                markersize=3, label=f'{emu_name} Pe')
        ax.plot(time_indices, Re_norm, marker='o', linewidth=1.5,
                markersize=3, linestyle='--', label=f'{emu_name} Re')

    ax.axhline(1, color='k', linewidth=0.8, linestyle=':')
    ax.grid(True, alpha=0.3)
    ax.set_ylabel(f'{var}\nNormalized MAE')

    # ==================================================
    # Depth-averaged arrays: expected shape = time, k, y, x
    # ==================================================
    llc_depth = as_dask(llc_patch[var].isel(k=slice(0, 51)))

    llc_t1_depth = llc_depth[:-1, :, :, :]
    llc_t2_depth = llc_depth[1:, :, :, :]

    baseline_k = da.nanmean(
        da.abs(llc_t2_depth - llc_t1_depth),
        axis=(-2, -1)
    )

    ax = axes[row, 1]

    for emu_name, emu_key in emulator_info:
        emu_depth = as_dask(emulator_patches[emu_key][var].isel(k=slice(0, 51)))
        emu_t2_depth = emu_depth[1:, :, :, :]

        Pe_k = da.nanmean(
            da.abs(llc_t2_depth - emu_t2_depth),
            axis=(-2, -1)
        )

        Re_k = da.nanmean(
            da.abs(llc_t1_depth - emu_t2_depth),
            axis=(-2, -1)
        )

        Pe_norm_depth = da.nanmean(safe_norm(Pe_k, baseline_k), axis=1)
        Re_norm_depth = da.nanmean(safe_norm(Re_k, baseline_k), axis=1)

        Pe_norm, Re_norm = dask.compute(Pe_norm_depth, Re_norm_depth)

        ax.plot(time_indices, Pe_norm, marker='o', linewidth=1.5,
                markersize=3, label=f'{emu_name} Pe')
        ax.plot(time_indices, Re_norm, marker='o', linewidth=1.5,
                markersize=3, linestyle='--', label=f'{emu_name} Re')

    ax.axhline(1, color='k', linewidth=0.8, linestyle=':')
    ax.grid(True, alpha=0.3)

    if row == 0:
        axes[row, 0].set_title('Surface', fontweight='bold')
        axes[row, 1].set_title('Depth-Averaged', fontweight='bold')

    if row == len(vars) - 1:
        axes[row, 0].set_xlabel('Time Step')
        axes[row, 1].set_xlabel('Time Step')

axes[0, 1].legend(fontsize=8, loc='best')

fig.suptitle('Prediction and Replication Errors by Variable', fontweight='bold')
plt.tight_layout()
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close()

Generating plots for Theta...
Generating plots for Salt...
